In [ ]:
"""
build_test_set.py
-----------------
Build a trustworthy held-out test set for the medical Q&A fine-tune.

Pipeline:
  1. Pull 100 random examples from MedQA (USMLE-style, human-written).
  2. Reshape them into the {"question": ..., "answer": ...} schema the
     fine-tune was trained on.
  3. Run an overlap check against the training set - both exact and fuzzy.
  4. Drop any leaked examples.
  5. Save to disk with a SHA-256 fingerprint so future loads are verifiable.

Usage:
  pip install datasets transformers --break-system-packages
  python build_test_set.py

Outputs:
  ./test_set_medqa_v1_<date>.jsonl                (one JSON object per line)
  ./test_set_medqa_v1_<date>.jsonl.manifest.json  (provenance + sha256)
"""

from __future__ import annotations

import hashlib
import json
import random
import re
from datetime import date
from difflib import SequenceMatcher
from pathlib import Path

from datasets import Dataset, load_dataset, load_from_disk  # noqa: F401

# --- Configuration -----------------------------------------------------------

RANDOM_SEED = 42
N_SAMPLES = 100
SIMILARITY_THRESHOLD = 0.80  # anything above this is treated as a near-duplicate

# Where your training set lives. Adjust to match your setup.
# Accepts:
#   - a local .jsonl / .json file   (e.g. "./medical_qa_synthetic_hf.jsonl")
#   - a local path saved via save_to_disk()
#   - a Hugging Face Hub dataset id (e.g. "lmoroney/medical-qa-v1")
TRAIN_SOURCE = "./medical_qa_synthetic_hf.jsonl"
TRAIN_SPLIT = "train"                            # only used for Hub datasets

# Where we'll write the final test set (single JSONL file).
OUT_FILE = Path(f"./test_set_medqa_v1_{date.today().isoformat()}.jsonl")

# --- Step 1: Pull MedQA ------------------------------------------------------

def load_medqa_sample(n: int, seed: int) -> Dataset:
    """Grab `n` random USMLE-style questions from MedQA.

    We use GBaker/MedQA-USMLE-4-options, a parquet-native mirror of the
    original MedQA USMLE set. Same questions, no loading script.
    """
    print("Loading MedQA (GBaker/MedQA-USMLE-4-options, test split)...")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
    print(f"  full test split: {len(ds)} examples")
    ds = ds.shuffle(seed=seed).select(range(n))
    print(f"  sampled: {len(ds)} examples")
    return ds


def medqa_to_qa(example: dict) -> dict:
    """
    GBaker/MedQA-USMLE-4-options example shape:
        {
          "question":   "<stem>",
          "answer":     "<correct option text>",
          "options":    {"A": "...", "B": "...", "C": "...", "D": "..."},
          "answer_idx": "A",
          "meta_info":  "step1" | "step2&3",
        }

    Some revisions of the dataset pack options as a list of {key, value}
    dicts instead of a plain dict - we handle both.

    We flatten to the {"question", "answer"} schema the model was trained
    on. We keep the option list inside the question so the task is
    well-specified, and the answer is the correct letter plus its
    expanded text.
    """
    opts = example["options"]
    if isinstance(opts, dict):
        items = sorted(opts.items())
    else:  # list of {"key": "...", "value": "..."}
        items = [(o["key"], o["value"]) for o in opts]

    options_block = "\n".join(f"{k}. {v}" for k, v in items)
    question = f"{example['question'].strip()}\n\nOptions:\n{options_block}"
    answer = f"{example['answer_idx']}. {example['answer'].strip()}"
    return {"question": question, "answer": answer}


# --- Step 2: Load training set for overlap check -----------------------------

def load_training_questions(source: str, split: str) -> list[str]:
    """Return a normalized list of training-set question strings.

    Supports:
      - local .jsonl / .json files (one example per line or a JSON array)
      - local HF datasets saved via save_to_disk()
      - Hugging Face Hub dataset ids
    """
    print(f"Loading training set from {source!r}...")
    src_path = Path(source)

    try:
        if src_path.is_file() and src_path.suffix.lower() in {".jsonl", ".json"}:
            ds = load_dataset("json", data_files=str(src_path), split="train")
        elif src_path.exists():
            ds = load_from_disk(source)
            if hasattr(ds, "keys") and split in ds:
                ds = ds[split]
        else:
            ds = load_dataset(source, split=split)
    except Exception as e:
        print(f"  WARNING: could not load training set ({e}).")
        print("  Overlap check will be skipped. You should fix this.")
        return []

    # Find the question field - tolerate a few common names.
    candidate_fields = ["question", "prompt", "input", "instruction", "query"]
    field = next((f for f in candidate_fields if f in ds.column_names), None)

    # Some synthetic datasets store everything in a "messages" list or a
    # single "text" field. Handle those too.
    if field is None and "messages" in ds.column_names:
        qs = []
        for msgs in ds["messages"]:
            user_turns = [m["content"] for m in msgs if m.get("role") == "user"]
            if user_turns:
                qs.append(user_turns[0].strip().lower())
        print(f"  training questions (from messages): {len(qs)}")
        return qs

    if field is None:
        print(f"  WARNING: no question-like column in {ds.column_names}.")
        return []

    qs = [str(q).strip().lower() for q in ds[field]]
    print(f"  training questions: {len(qs)}")
    return qs


# --- Step 3: Overlap detection -----------------------------------------------

_TOKEN_RE = re.compile(r"\b[a-z]{4,}\b")

def _shingles(text: str, k: int = 4) -> set[tuple]:
    """k-word shingles for cheap near-duplicate prefilter."""
    words = _TOKEN_RE.findall(text.lower())
    return {tuple(words[i:i + k]) for i in range(len(words) - k + 1)}


def find_overlaps(
    test_qs: list[str],
    train_qs: list[str],
    threshold: float,
) -> tuple[set[int], set[int]]:
    """
    Return (exact_idx, near_idx) - indices into test_qs that should be dropped.

    We use a shingle-based prefilter so we don't run SequenceMatcher
    over N*M pairs. Any test question sharing >=1 4-word shingle with
    any training question gets a full ratio check.
    """
    if not train_qs:
        return set(), set()

    train_set = set(train_qs)
    train_shingles_by_idx = [_shingles(q) for q in train_qs]
    # invert: shingle -> list of train indices that contain it
    shingle_index: dict[tuple, list[int]] = {}
    for i, sh in enumerate(train_shingles_by_idx):
        for s in sh:
            shingle_index.setdefault(s, []).append(i)

    exact: set[int] = set()
    near: set[int] = set()

    for ti, tq in enumerate(test_qs):
        if tq in train_set:
            exact.add(ti)
            continue

        # Candidates = any train question sharing a 4-gram
        candidate_idxs: set[int] = set()
        for s in _shingles(tq):
            candidate_idxs.update(shingle_index.get(s, []))

        for ci in candidate_idxs:
            ratio = SequenceMatcher(None, tq, train_qs[ci]).ratio()
            if ratio >= threshold:
                near.add(ti)
                break

    return exact, near


# --- Step 4 & 5: Save with fingerprint ---------------------------------------

def save_test_set(ds: Dataset, out_file: Path, manifest: dict) -> None:
    """Write the test set as a JSONL file plus a sidecar manifest."""
    out_file.parent.mkdir(parents=True, exist_ok=True)

    rows = [{"question": ex["question"], "answer": ex["answer"]} for ex in ds]

    # One JSON object per line.
    with out_file.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # Fingerprint the exact text we'll evaluate on.
    payload = json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()
    fingerprint = hashlib.sha256(payload).hexdigest()

    manifest["fingerprint_sha256"] = fingerprint
    manifest_path = out_file.with_suffix(out_file.suffix + ".manifest.json")
    manifest_path.write_text(json.dumps(manifest, indent=2))

    print(f"\nSaved test set to: {out_file}")
    print(f"  manifest:    {manifest_path}")
    print(f"  fingerprint: {fingerprint[:16]}...")
    print(f"  examples:    {len(rows)}")


# --- Main --------------------------------------------------------------------

def main() -> None:
    random.seed(RANDOM_SEED)

    # 1. Pull MedQA
    raw = load_medqa_sample(N_SAMPLES, RANDOM_SEED)

    # 2. Reshape to our Q&A schema
    test = raw.map(medqa_to_qa, remove_columns=raw.column_names)

    # 3. Overlap check against training set
    train_qs = load_training_questions(TRAIN_SOURCE, TRAIN_SPLIT)
    test_qs_norm = [ex["question"].strip().lower() for ex in test]

    print(f"\nChecking for leakage against {len(train_qs)} training questions...")
    exact, near = find_overlaps(test_qs_norm, train_qs, SIMILARITY_THRESHOLD)
    print(f"  exact duplicates:      {len(exact)}")
    print(f"  near-duplicates (>{int(SIMILARITY_THRESHOLD * 100)}%): {len(near)}")

    # 4. Drop any leaked examples
    to_drop = exact | near
    if to_drop:
        keep_idx = [i for i in range(len(test)) if i not in to_drop]
        print(f"  dropping {len(to_drop)} leaked example(s); {len(keep_idx)} remain")
        test = test.select(keep_idx)
    else:
        print("  no leakage detected")

    # 5. Save with fingerprint + manifest
    manifest = {
        "created": date.today().isoformat(),
        "source": "bigbio/med_qa (en_source, test split)",
        "n_sampled": N_SAMPLES,
        "n_after_leakage_filter": len(test),
        "exact_overlap_dropped": len(exact),
        "near_overlap_dropped": len(near),
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "train_source": TRAIN_SOURCE,
        "seed": RANDOM_SEED,
    }
    save_test_set(test, OUT_FILE, manifest)

    # Peek at one example so you can see what you're evaluating on.
    print("\nSample example:")
    print("-" * 60)
    print("Q:", test[0]["question"][:300], "...")
    print("A:", test[0]["answer"])
    print("-" * 60)


if __name__ == "__main__":
    main()


Loading MedQA (GBaker/MedQA-USMLE-4-options, test split)...
  full test split: 1273 examples
  sampled: 100 examples
Loading training set from './medical_qa_synthetic_hf.jsonl'...
  training questions: 611

Checking for leakage against 611 training questions...
  exact duplicates:      0
  near-duplicates (>80%): 0
  no leakage detected

Saved test set to: test_set_medqa_v1_2026-04-14.jsonl
  manifest:    test_set_medqa_v1_2026-04-14.jsonl.manifest.json
  fingerprint: 4e3d4f334d900a4b...
  examples:    100

Sample example:
------------------------------------------------------------
Q: A 6-year-old African American boy is referred to the hospital by his family physician for jaundice, normocytic anemia, and severe bone pain. He has a history of several episodes of mild bone pain in the past treated with over the counter analgesics. On physical examination, the child is icteric with ...
A: B. Sickle cell disease
------------------------------------------------------------
